# MWIZERWA AI — Google Colab

Run cells **in order** with **Runtime → GPU**.

| Step | Purpose |
|------|---------|
| 1 | Clone/use `https://github.com/Mindora-AI/ai_engine` at `/content/ai_engine` |
| 2 | Mount Drive → `MyDrive/MWIZERWA` |
| 3 | Hardware inspection |
| 4 | Python dependencies |
| 5 | CUDA-aware `llama-cpp-python` + GPU layers |
| 6 | DeepKIN (KinyaFlex) |
| 7 | Model check & download |
| 8 | Real validation |
| 9 | Gradio UI (only if validation passes) |

Private repo: set env `MWIZERWA_REPO_URL` before step 1 (use Colab Secrets, not hardcoded tokens).

Docs: `docs/COLAB.md`

## 1. Repository initialization

Success: prints path to existing checkout or clones into `/content/ai_engine`.

Failure: non-empty folder without `run_assistant.py` — rename/remove manually.

In [1]:
import os
import sys
from pathlib import Path

PROJECT = Path(os.environ.get('MWIZERWA_COLAB_PROJECT', '/content/ai_engine'))
MARKER = PROJECT / 'run_assistant.py'

if not MARKER.is_file():
    if PROJECT.is_file():
        raise RuntimeError(f'{PROJECT} is a file, not a directory')
    if PROJECT.exists() and any(PROJECT.iterdir()):
        raise RuntimeError(
            f'{PROJECT} exists but is not a MWIZERWA checkout. '
            'Remove/rename it or set MWIZERWA_COLAB_PROJECT.'
        )
    import subprocess
    url = os.environ.get('MWIZERWA_REPO_URL', 'https://github.com/Mindora-AI/ai_engine.git')
    PROJECT.parent.mkdir(parents=True, exist_ok=True)
    print('Cloning', url)
    subprocess.run(['git', 'clone', '--depth', '1', url, str(PROJECT)], check=True)
else:
    print('Using existing project:', PROJECT)
    import subprocess
    subprocess.run(['git', 'pull', '--ff-only', 'origin', 'main'], cwd=PROJECT)

os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
os.environ['MWIZERWA_PROJECT_ROOT'] = str(PROJECT)

from scripts.colab_runtime import require_colab_scripts
require_colab_scripts(PROJECT)
print('MWIZERWA_PROJECT_ROOT =', PROJECT)

Using existing project: /content/ai_engine
MWIZERWA_PROJECT_ROOT = /content/ai_engine


## 2. Google Drive persistence

Success: env vars `MWIZERWA_MODELS_DIR`, `MWIZERWA_CACHE_DIR`, `MWIZERWA_OUTPUT_DIR`, `HF_HOME` printed.

If `drive.mount` fails, fix authentication here — later cells will not recover.

Creating folders does **not** mean model files exist yet (step 7).

In [2]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/MWIZERWA')

from scripts.colab_runtime import configure_drive_persistence, report_model_asset_status

configure_drive_persistence(DRIVE_ROOT, PROJECT)
report_model_asset_status()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[Drive] Expected model tree (files may still be MISSING until download):
  models/afriqueqwen  exists=False
  models/kinya-flex-tts  exists=False
  models/piper/en_US-ljspeech-medium  exists=False
  models/piper/fr_FR-siwis-medium  exists=False
[Config] MWIZERWA_PROJECT_ROOT=/content/ai_engine (exists=True)
[Config] MWIZERWA_MODELS_DIR=/content/drive/MyDrive/MWIZERWA/models (exists=True)
[Config] MWIZERWA_CACHE_DIR=/content/drive/MyDrive/MWIZERWA/cache (exists=True)
[Config] MWIZERWA_OUTPUT_DIR=/content/drive/MyDrive/MWIZERWA/outputs (exists=True)
[Config] HF_HOME=/content/drive/MyDrive/MWIZERWA/cache/huggingface (exists=True)
[Models] Asset status (file must exist before inference):
  MISSING llm_gguf: /content/drive/MyDrive/MWIZERWA/models/afriqueqwen/AfriqueQwen3.5-4B-Instruct-v1.i1-Q4_K_M.gguf
  MISSING kinya_flex: /content/drive/MyDrive/MWIZERWA/models/k

{'llm_gguf': False, 'kinya_flex': False, 'piper_en': False, 'piper_fr': False}

## 3. Hardware inspection

Inspect RAM, disk, and GPU before installing heavy packages.

In [3]:
import json
from scripts.colab_runtime import inspect_hardware

HW = inspect_hardware()
print(json.dumps(HW, indent=2))

{
  "platform": "Linux-6.6.159+-x86_64-with-glibc2.39",
  "python": "3.13.16",
  "colab": true,
  "cpu_count": 2,
  "ram_gb": 12.67,
  "ram_available_gb": 11.44,
  "disk_free_gb": 110.03,
  "torch_cuda_version": "13.0",
  "cuda_available": true,
  "gpu_name": "Tesla T4",
  "gpu_total_mem_gb": 14.56
}


## 4. Dependencies

Installs `requirements-colab.txt`. **llama-cpp** is installed in the next cell (CUDA-aware).

In [4]:
from scripts.colab_runtime import install_python_dependencies

install_python_dependencies(PROJECT)

[Deps] Keeping existing torch 2.14.1+cu130 (cuda=True)
[pip] requirements-colab: install -r /content/ai_engine/requirements-colab.txt -c /content/ai_engine/constraints-colab.txt
[Deps] Base packages installed (Gradio installed next).
[Deps] Upgrading Gradio (hub 1.x removed HfFolder; Gradio 4.x cannot import). Current gradio=5.50.0, huggingface_hub major=1.
[pip] uninstall old gradio: uninstall -y gradio
Found existing installation: gradio 5.50.0
Uninstalling gradio-5.50.0:
  Successfully uninstalled gradio-5.50.0
[pip] install pinned gradio: install gradio==5.50.0
  Using cached gradio-5.50.0-py3-none-any.whl.metadata (16 kB)
Using cached gradio-5.50.0-py3-none-any.whl (63.5 MB)
[Deps] Verified gradio import: 5.50.0 1.33.0


## 5. llama-cpp-python (CUDA when available)

Tries a wheel matching `torch.version.cuda`, then fallbacks `cu124`…`cu121`.

- Success (CUDA): message `CUDA offload available`.
- CPU fallback: explicit warning; `MWIZERWA_LLM_N_GPU_LAYERS=0`.

In [5]:
from scripts.colab_runtime import (
    install_llama_cpp_for_colab,
    apply_llm_gpu_layers,
    llama_supports_gpu_offload,
)

LLAMA_INFO = install_llama_cpp_for_colab()
print(LLAMA_INFO)
print('llama_supports_gpu_offload():', llama_supports_gpu_offload())
apply_llm_gpu_layers(HW, LLAMA_INFO)

[llama-cpp] Trying CUDA wheel index: https://abetlen.github.io/llama-cpp-python/whl/cu130


ggml_cuda_init: found 1 CUDA devices (Total VRAM: 14912 MiB):
  Device 0: Tesla T4, compute capability 7.5, VMM: yes, VRAM: 14912 MiB


[llama-cpp] CUDA offload available (wheel tag cu130).
{'backend': 'cuda', 'wheel_tag': 'cu130', 'gpu_offload_supported': True}
llama_supports_gpu_offload(): True
[Config] MWIZERWA_LLM_N_GPU_LAYERS=24


24

In [6]:
import sys
import subprocess

subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "--upgrade",
    "typed-argument-parser",
    "numpy<2.5"
])

print("Installation complete. Restart the Colab runtime before continuing.")

Installation complete. Restart the Colab runtime before continuing.


## 6. DeepKIN (KinyaFlex)

Required for Kinyarwanda TTS only.

In [7]:
from scripts.colab_runtime import (
    ensure_gradio_hub_compatibility,
    install_deepkin,
    verify_critical_imports,
)

ensure_gradio_hub_compatibility(force=True)
install_deepkin(PROJECT)
verify_critical_imports()

[Deps] Upgrading Gradio (hub 1.x removed HfFolder; Gradio 4.x cannot import). Current gradio=5.50.0, huggingface_hub major=1.
[pip] uninstall old gradio: uninstall -y gradio
Found existing installation: gradio 5.50.0
Uninstalling gradio-5.50.0:
  Successfully uninstalled gradio-5.50.0
[pip] install pinned gradio: install gradio==5.50.0
  Using cached gradio-5.50.0-py3-none-any.whl.metadata (16 kB)
Using cached gradio-5.50.0-py3-none-any.whl (63.5 MB)
[Deps] Verified gradio import: 5.50.0 1.33.0
[DeepKIN] deepkin already importable — skipping reinstall.
[Imports] Interpreter: /usr/bin/python3 | hub=1.33.0 | gradio(dist)=5.50.0 | HfFolder=False
[Import] OK   torch                v=2.14.1+cu130 (required)
[Import] OK   transformers         v=5.22.1       (required)
[Import] OK   llama-cpp-python     v=0.3.37       (required)
[Import] OK   gradio               v=5.50.0       (required)
[Import] OK   deepkin              v=1.0.0        (required)
[Import] OK   huggingface_hub      v=1.33.0 

[{'package': 'torch',
  'import': 'import torch',
  'required': True,
  'ok': True,
  'version': '2.14.1+cu130',
  'error': None},
 {'package': 'transformers',
  'import': 'import transformers',
  'required': True,
  'ok': True,
  'version': '5.22.1',
  'error': None},
 {'package': 'llama-cpp-python',
  'import': 'from llama_cpp import Llama',
  'required': True,
  'ok': True,
  'version': '0.3.37',
  'error': None},
 {'package': 'gradio',
  'import': 'import gradio',
  'required': True,
  'ok': True,
  'version': '5.50.0',
  'error': None},
 {'package': 'deepkin',
  'import': 'import deepkin',
  'required': True,
  'ok': True,
  'version': '1.0.0',
  'error': None},
 {'package': 'huggingface_hub',
  'import': 'import huggingface_hub',
  'required': True,
  'ok': True,
  'version': '1.33.0',
  'error': None},
 {'package': 'onnxruntime',
  'import': 'import onnxruntime',
  'required': True,
  'ok': True,
  'version': '1.32.0',
  'error': None}]

## 7. Models — verify then download

`--check` must pass before download (no `|| true`).

Reuses valid files on Drive; downloads only missing/invalid assets.

In [11]:
import subprocess
import sys

def run_py(args, allow_failure=False):
    print("\n>>", " ".join(args), flush=True)
    result = subprocess.run(
        args,
        cwd=PROJECT,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT
    )
    print(result.stdout, flush=True)

    if result.returncode != 0 and not allow_failure:
        raise RuntimeError(f"Command failed with exit code {result.returncode}")

    return result.returncode

py = sys.executable

# Initial check: missing files are expected.
run_py([py, "scripts/download_models.py", "--check"], allow_failure=True)

# Download LLM and KinyaFlex.
run_py([py, "scripts/download_models.py"])

# Download both Piper voices.
run_py([py, "scripts/download_models.py", "--model", "piper"])

# Final verification: must pass.
run_py([py, "scripts/download_models.py", "--check"])

from scripts.colab_runtime import report_model_asset_status
report_model_asset_status()


>> /usr/bin/python3 scripts/download_models.py --check
Models root: /content/drive/MyDrive/MWIZERWA/models

MWIZERWA AI — Model Checkpoint Status

Model: AfriqueQwen3.5-4B-Instruct GGUF
  Location:  /content/drive/MyDrive/MWIZERWA/models/afriqueqwen/AfriqueQwen3.5-4B-Instruct-v1.i1-Q4_K_M.gguf
  Present:   NO
  Status:    MISSING

Model: KinyaFlex TTS Base Checkpoint
  Location:  /content/drive/MyDrive/MWIZERWA/models/kinya-flex-tts/kinya_flex_tts_base_trained.pt
  Present:   NO
  Status:    MISSING

Model: Piper TTS — English (en_US-ljspeech-medium)
  File:     /content/drive/MyDrive/MWIZERWA/models/piper/en_US-ljspeech-medium/en_US-ljspeech-medium.onnx
  Present:  NO
  File:     /content/drive/MyDrive/MWIZERWA/models/piper/en_US-ljspeech-medium/en_US-ljspeech-medium.onnx.json
  Present:  NO
  Bundle:   MISSING

Model: Piper TTS — French (fr_FR-siwis-medium)
  File:     /content/drive/MyDrive/MWIZERWA/models/piper/fr_FR-siwis-medium/fr_FR-siwis-medium.onnx
  Present:  NO
  File:     

{'llm_gguf': True, 'kinya_flex': True, 'piper_en': True, 'piper_fr': True}

## 8. Real validation

LLM (rw/en/fr), KinyaFlex + Piper TTS, optional ASR.

Must exit 0 before Gradio.

In [12]:
import subprocess
import sys

r = subprocess.run([sys.executable, 'scripts/colab_validate.py'], cwd=PROJECT)
if r.returncode != 0:
    raise RuntimeError(
        'Validation failed. Fix models/deps above — Gradio will not start.'
    )
MWIZERWA_VALIDATION_OK = True
print('Validation OK — safe to launch Gradio.')

Validation OK — safe to launch Gradio.


## 9. Gradio UI

Uses `ConversationEngine`, safety gates, and TTS router (KinyaFlex/Piper).

Audio on Drive: `MyDrive/MWIZERWA/outputs/tts_responses/latest_response.wav`

In [1]:
import subprocess
import sys

def run(args):
    print(f"\n$ {' '.join(args)}", flush=True)
    result = subprocess.run(
        args,
        cwd="/content/ai_engine",
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT
    )
    print(result.stdout, flush=True)

    if result.returncode != 0:
        raise RuntimeError(f"Command failed: {' '.join(args)}")

py = sys.executable

# Download the main model assets and Piper voices.
run([py, "scripts/download_models.py"])
run([py, "scripts/download_models.py", "--model", "piper"])

# One final verification.
run([py, "scripts/download_models.py", "--check"])


$ /usr/bin/python3 scripts/download_models.py
/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_validators.py:206: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
Models root: /content/ai_engine/models

MWIZERWA AI — Model Checkpoint Status

Model: AfriqueQwen3.5-4B-Instruct GGUF
  Location:  /content/ai_engine/models/afriqueqwen/AfriqueQwen3.5-4B-Instruct-v1.i1-Q4_K_M.gguf
  Present:   NO
  Status:    MISSING

Model: KinyaFlex TTS Base Checkpoint
  Location:  /content/ai_engine/models/kinya-flex-tts/kinya_flex_tts_base_trained.pt
  Present:   NO
  Status:    MISSING

Model: Piper TTS — English (en_US-ljspeech-medium)
  File:     /content/ai_engine/models/piper/en_US-ljspeech-medium/en_US-ljspeech-medium.onnx
  Present:  NO
  File:     /content/ai_engine/models/piper/en_US-ljspeech-medium/en_US-ljspeech-medium.onnx.json
  Present:  NO
  Bundl

In [2]:
import sys
import inspect
import torch

PROJECT = "/content/ai_engine"
if PROJECT not in sys.path:
    sys.path.insert(0, PROJECT)

from src.conversation.engine import ConversationEngine

print("Engine import: SUCCESS")
print("Constructor:", inspect.signature(ConversationEngine))
print("Public methods:")

for name, method in inspect.getmembers(ConversationEngine, predicate=inspect.isfunction):
    if not name.startswith("_"):
        try:
            print(f"  {name}{inspect.signature(method)}")
        except (TypeError, ValueError):
            print(f"  {name}")

print("\nGPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Unavailable")

Engine import: SUCCESS
Constructor: (manager: Optional[src.llm.conversation.ConversationManager] = None, state: Optional[src.conversation.session.SessionState] = None, language_detector: Optional[src.utils.language_detector.LanguageDetector] = None, safety_evaluator: Optional[src.safety.evaluator.SafetyEvaluator] = None, safety_guardrail: Optional[src.safety.guardrail.SafetyGuardrail] = None, assistant_validator: Optional[src.safety.assistant_policy.AssistantSafetyValidator] = None, recommender: Optional[src.exercises.recommender.ExerciseRecommender] = None, lifecycle: Optional[src.exercises.lifecycle.ExerciseLifecycle] = None)
Public methods:
  process_turn(self, request: src.conversation.session.TurnRequest) -> src.conversation.session.TurnResult
  reset_conversation(self) -> None

GPU: Tesla T4


In [3]:
import inspect
from src.llm.conversation import ConversationManager

print("ConversationManager constructor:")
print(inspect.signature(ConversationManager))

print("\nConversationManager initialization:")
print(inspect.getsource(ConversationManager.__init__))

print("\nRelevant model-loading methods:")
for name, method in inspect.getmembers(ConversationManager, predicate=inspect.isfunction):
    if any(word in name.lower() for word in ("load", "init", "model", "generate", "chat")):
        try:
            print(f"\n{name}{inspect.signature(method)}")
        except (TypeError, ValueError):
            print(name)

ConversationManager constructor:
(llm: Optional[src.llm.afrique_qwen.AfriqueQwenLLM] = None, max_turns: Optional[int] = None, n_ctx: Optional[int] = None, max_tokens: Optional[int] = None, safety_margin: int = 16, system_prompt: str = '\nYou are MWIZERWA AI, a supportive, multilingual wellness companion.\nYour purpose is to listen carefully, understand what the user is communicating, recognize their emotional context without jumping to conclusions, help them reflect when appropriate, and offer practical suggestions when requested or genuinely useful.\n\nYou communicate naturally, warmly, and respectfully.\n\nCRITICAL IDENTITY BOUNDARY:\nYou are an AI companion. You are NOT a human, therapist, psychologist, psychiatrist, doctor, or healthcare professional. You do not provide clinical therapy, medical advice, or psychiatric diagnosis.\n\n1. CONVERSATIONAL UNDERSTANDING & INTENTIONS\n\nFor every turn, determine the user\'s primary conversational intention:\n- "LISTENING": The user primari

In [4]:
from src.llm.conversation import ConversationManager
from src.conversation.engine import ConversationEngine

print("Initializing ConversationManager and loading AfriqueQwen...")
manager = ConversationManager()

print("Initializing MWIZERWA ConversationEngine...")
engine = ConversationEngine(manager=manager)

print("Running inference test...")
response = manager.generate_response(
    "Hello. Briefly introduce yourself."
)

print("\nMWIZERWA RESPONSE:")
print(response)

print("\nENGINE INITIALIZATION AND INFERENCE TEST COMPLETE.")

Initializing ConversationManager and loading AfriqueQwen...
Initializing MWIZERWA ConversationEngine...
Running inference test...
Loading AfriqueQwen model from: /content/ai_engine/models/afriqueqwen/AfriqueQwen3.5-4B-Instruct-v1.i1-Q4_K_M.gguf ...
AfriqueQwen model loaded successfully!

MWIZERWA RESPONSE:
Hello! I'm ChatGPT, an AI developed by OpenAI. I'm designed to assist users with a wide range of tasks, from answering questions to providing information and engaging in conversation.

ENGINE INITIALIZATION AND INFERENCE TEST COMPLETE.


In [5]:
import subprocess
import sys
import os

PROJECT = "/content/ai_engine"

def run(args):
    result = subprocess.run(
        args,
        cwd=PROJECT,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT
    )
    print(result.stdout)
    if result.returncode:
        raise RuntimeError("Command failed: " + " ".join(args))

run(["git", "pull", "origin", "main"])

run([
    sys.executable, "-m", "pip",
    "install", "-r", "requirements-api.txt"
])

print("API files updated and dependencies installed.")

From https://github.com/Mindora-AI/ai_engine
 * branch            main       -> FETCH_HEAD
   a97993b..2992131  main       -> origin/main
Updating a97993b..2992131
Fast-forward
 requirements-api.txt |   3 ++
 scripts/serve_api.py |  29 +++++++++++
 src/api/__init__.py  |   5 ++
 src/api/app.py       |  87 ++++++++++++++++++++++++++++++++
 src/api/schemas.py   |  45 +++++++++++++++++
 src/api/service.py   | 140 +++++++++++++++++++++++++++++++++++++++++++++++++++
 tests/test_api.py    | 105 ++++++++++++++++++++++++++++++++++++++
 7 files changed, 414 insertions(+)
 create mode 100644 requirements-api.txt
 create mode 100644 scripts/serve_api.py
 create mode 100644 src/api/__init__.py
 create mode 100644 src/api/app.py
 create mode 100644 src/api/schemas.py
 create mode 100644 src/api/service.py
 create mode 100644 tests/test_api.py


API files updated and dependencies installed.


In [10]:
!pip install pyngrok

In [13]:
from pyngrok import ngrok
from getpass import getpass

token = getpass("Enter your ngrok authtoken: ")
ngrok.set_auth_token(token)

tunnel = ngrok.connect(8000, "http")
print("Public API URL:", tunnel.public_url)

Enter your ngrok authtoken: ··········
Public API URL: https://rephrase-fiber-dork.ngrok-free.dev


In [ ]:
%cd /content/ai_engine
!python scripts/serve_api.py

/content/ai_engine
INFO:     Started server process [49917]
INFO:     Waiting for application startup.
Loading AfriqueQwen model from: /content/ai_engine/models/afriqueqwen/AfriqueQwen3.5-4B-Instruct-v1.i1-Q4_K_M.gguf ...
AfriqueQwen model loaded successfully!
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)
INFO:     197.243.56.142:0 - "GET /health HTTP/1.1" 200 OK
INFO:     197.243.56.142:0 - "GET /favicon.ico HTTP/1.1" 404 Not Found
INFO:     197.243.56.142:0 - "GET /docs HTTP/1.1" 200 OK
INFO:     197.243.56.142:0 - "GET /openapi.json HTTP/1.1" 200 OK
INFO:     197.243.56.142:0 - "POST /chat HTTP/1.1" 200 OK
